# บทที่ 8: การนำเข้าข้อมูลด้วย pandas

ในบทก่อนหน้า เราได้เรียนรู้การสร้างและใช้งาน pandas `Series` และ `DataFrame`

แต่ในการทำงานจริง นักวิเคราะห์ข้อมูลมักไม่ได้สร้าง DataFrame ด้วยตนเอง ข้อมูลส่วนใหญ่มาจากแหล่งต่าง ๆ เช่น

- ไฟล์ CSV
- ไฟล์ Excel
- Excel ที่มีหลาย Worksheet
- ไฟล์ข้อความที่มีตัวคั่น
- ไฟล์ที่เผยแพร่ผ่าน URL

กระบวนการเปลี่ยนข้อมูลจากแหล่งต้นทางให้เป็น DataFrame เรียกว่า **data ingestion**

เป้าหมายของบทนี้ไม่ใช่เพียงทำให้ pandas เปิดไฟล์ได้ แต่ต้องนำเข้าข้อมูลให้

- ใช้ชื่อคอลัมน์ถูกต้อง
- รักษารหัสที่มีเลขศูนย์นำหน้า
- อ่านวันที่เป็นชนิดข้อมูลที่เหมาะสม
- เลือกเฉพาะข้อมูลที่จำเป็น
- ตรวจสอบโครงสร้างหลังนำเข้า
- นำขั้นตอนกลับมาใช้ซ้ำได้

## ผลการเรียนรู้ที่คาดหวัง

เมื่อเรียนจบบทนี้ ผู้เรียนจะสามารถ

1. นำเข้าข้อมูลจาก CSV, Excel และไฟล์ข้อความด้วย pandas ได้
2. กำหนด parameter สำคัญ เช่น `encoding`, `usecols`, `dtype`, `parse_dates`, `sheet_name` และ `sep` ได้
3. อ่านและรวมข้อมูลจากหลาย Worksheet ได้อย่างเหมาะสม
4. ตรวจสอบความถูกต้องของ DataFrame หลังการนำเข้าได้

## ลำดับเนื้อหา

บทเรียนนี้ประกอบด้วยหัวข้อต่อไปนี้

1. เตรียม pandas และตำแหน่งไฟล์
2. ตรวจสอบไฟล์ก่อนนำเข้า
3. ทดลองอ่านไฟล์ CSV
4. กำหนด `encoding`
5. อ่านบางแถวด้วย `nrows`
6. เลือกคอลัมน์ด้วย `usecols`
7. กำหนดชนิดข้อมูลด้วย `dtype`
8. อ่านวันที่ด้วย `parse_dates`
9. ตรวจสอบ DataFrame หลังนำเข้า
10. อ่านไฟล์ Excel
11. ตรวจสอบและเลือก Worksheet
12. อ่านหลาย Worksheet
13. รวมข้อมูลจากหลาย Worksheet
14. อ่านไฟล์จาก URL
15. อ่านไฟล์ข้อความที่มีตัวคั่น
16. กำหนดชื่อคอลัมน์ให้ไฟล์ที่ไม่มี Header
17. ตรวจสอบความสอดคล้องของโครงสร้าง
18. ข้อผิดพลาดที่พบบ่อย
19. แบบฝึกหัดท้ายบท

## 1. เตรียม pandas และตำแหน่งไฟล์

ก่อนนำเข้าข้อมูล ต้อง import pandas

ในบทนี้จะใช้ `Path` จาก Library มาตรฐานของ Python เพื่อจัดการตำแหน่งไฟล์

ข้อดีของ `Path` ได้แก่

- อ่านตำแหน่งไฟล์ได้ง่าย
- ตรวจสอบว่าไฟล์มีอยู่จริงหรือไม่
- ใช้งานได้สะดวกกว่าการต่อข้อความ Path ด้วยตนเอง

In [1]:
from pathlib import Path

import pandas as pd

กำหนดชื่อไฟล์ตัวอย่างที่ใช้ในบทเรียน

In [2]:
csv_file = Path(
    "msdhs_ops_mso_logbook.csv"
)

excel_file = Path(
    "moac_opsmoac_fragile_farmer.xlsx"
)

population_url = (
    "https://stat.bora.dopa.go.th/"
    "new_stat/file/6812/6812cc10.txt"
)

ตำแหน่งไฟล์แบบ Relative Path จะอ้างอิงจาก Working Directory ปัจจุบันของ Notebook

สามารถตรวจสอบ Working Directory ได้ด้วย

In [3]:
Path.cwd()

PosixPath('/workspaces/MSDHS_OJT/py-jupyter_docker/course/day3')

หาก Notebook กับไฟล์ข้อมูลอยู่ใน Folder เดียวกัน การใช้เพียงชื่อไฟล์มักเพียงพอ

หากไฟล์อยู่ใน Folder ย่อย สามารถเขียนว่า

```python
data_dir = Path("data")

csv_file = (
    data_dir
    / "msdhs_ops_mso_logbook.csv"
)
```

## 2. ตรวจสอบไฟล์ก่อนนำเข้า

ก่อนใช้ `pd.read_csv()` หรือ `pd.read_excel()` ควรตรวจสอบว่า

1. Path มีอยู่จริง
2. Path นั้นเป็นไฟล์
3. ชื่อและนามสกุลไฟล์ถูกต้อง

การตรวจสอบล่วงหน้าช่วยแยกปัญหาเรื่องตำแหน่งไฟล์ออกจากปัญหาเรื่องรูปแบบข้อมูล

In [4]:
print("CSV exists:", csv_file.exists())
print("CSV is file:", csv_file.is_file())

print(
    "Excel exists:",
    excel_file.exists(),
)

print(
    "Excel is file:",
    excel_file.is_file(),
)

CSV exists: True
CSV is file: True
Excel exists: True
Excel is file: True


หากได้ `False` ควรตรวจสอบ

- Working Directory
- ชื่อไฟล์
- นามสกุลไฟล์
- ตัวพิมพ์เล็กและใหญ่
- Folder ที่เก็บไฟล์

ไม่ควรแก้ปัญหาด้วยการเปลี่ยน parameter ของ pandas จนกว่าจะยืนยันว่า Path ถูกต้อง

## 3. ทดลองอ่านไฟล์ CSV

คำสั่งหลักสำหรับอ่าน CSV คือ

```python
pd.read_csv()
```

รูปแบบพื้นฐานคือ

```python
dataframe = pd.read_csv(
    "file_name.csv"
)
```

อย่างไรก็ตาม ไม่ควรเริ่มจากการอ่านไฟล์ขนาดใหญ่ทั้งหมดทันที

แนวทางที่แนะนำคือ

1. ทดลองอ่านเพียงบางแถว
2. ตรวจสอบชื่อคอลัมน์
3. ตรวจสอบชนิดข้อมูล
4. กำหนด parameter ที่จำเป็น
5. อ่านข้อมูลสำหรับใช้งานจริง

In [5]:
mso_preview = pd.read_csv(
    csv_file,
    nrows=5,
)

mso_preview

,วันที่แก้ไขข้อมูลล่าสุด,ชื่อสสว.,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,ลักษณะที่อยู่อาศัย,เลขบัตรประชาชน,เลขที่สมาชิกครัวเรือน,คำนำหน้า,ชื่อ,...,ปัญหาด้านการเป็นผู้เสียหายจากการค้ามนุษย์,ปัญหาด้านการเข้าไม่ถึงสิทธิและความเป็นธรรมในสังคม,ปัญหาด้านสภาพปัญหาสังคมอื่นๆ,ประเภทกลุ่มเป้าหมาย,รหัส cm,ชื่อ นามสกุล cm,หน่วยงานของ cm,ละติจูด,ลองจิจูด,สถานะการมีชีวิต
0,20230619 13:40:56.585,5,648ff7d6b64d06b6b0dc7f02,NaN,20230619 13:38:14.881,ที่ดินมีกรรมสิทธิ์ของตนเอง,5962222985aa759afbc332b96975a4c4d74a5f82a1ab8a...,648ff83350573a1fb581f155,เด็กหญิง,****,...,NaN,NaN,NaN,เด็กเล็ก,cm410011,****,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...,****,****,1
1,20220717 18:56:10.147,3,62d3f7a6d6f101550054198b,7.004022e+10,20220717 18:51:02.889,ที่ดินของผู้อื่น,62252665a08c86140a798453916f866d01ff17b8683341...,62d3f8aa24e6c5bf7e6edfdf,นางสาว,****,...,NaN,NaN,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm700018,****,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี,****,****,1
2,20240125 13:34:46.941,5,62a05c9df5674ecdd3805787,NaN,20220806 15:23:57.850,ที่ดินมีกรรมสิทธิ์ของตนเอง,761904ae1beb77d5de38fcbf34e757020c89291e467225...,62a05c9d63390df4b203ffdc,นางสาว,****,...,NaN,NaN,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm430010,****,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...,****,****,1
3,20240322 14:12:39.290,6,62908e378fa67bf5ad7d5555,NaN,20220527 15:39:19.197,ที่ดินมีกรรมสิทธิ์ของตนเอง,a94f022ed6a769f28d8cd2eefafed33d4185008de20a0b...,62908e3763390df4b20378f6,เด็กหญิง,****,...,NaN,ไม่สามารถเข้าถึงบริการของรัฐ|,NaN,เด็กเล็ก,cm340009,****,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...,****,****,1
4,20230315 09:27:11.763,4,6319b26ca3a37508384e36fc,NaN,20220809 16:14:20.652,ที่ดินมีกรรมสิทธิ์ของตนเอง,704921c77c739b12a7551cadae9998e06e50848647c058...,64112cc9dafc21d42cd4cfac,เด็กชาย,****,...,NaN,ไม่สามารถเข้าถึงบริการของรัฐ|,NaN,เด็กเล็ก,cm300020,****,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา,****,****,1


หากไฟล์ภาษาไทยแสดงผลถูกต้อง สามารถอ่านต่อได้

หากตัวอักษรผิดปกติหรือเกิด `UnicodeDecodeError` ต้องตรวจสอบ `encoding`

### ตรวจสอบโครงสร้างจากตัวอย่าง

In [6]:
print(mso_preview.shape)

mso_preview.columns.tolist()

(5, 50)


['วันที่แก้ไขข้อมูลล่าสุด',
 'ชื่อสสว.',
 'รหัสครัวเรือน',
 'รหัสประจำบ้าน',
 'วันที่สร้างครัวเรือน',
 'ลักษณะที่อยู่อาศัย',
 'เลขบัตรประชาชน',
 'เลขที่สมาชิกครัวเรือน',
 'คำนำหน้า',
 'ชื่อ',
 'นามสกุล',
 'วันเดือนปีเกิด',
 'อายุ',
 'เพศ',
 'สัญชาติ',
 'สัญชาติอื่น (ระบุ)',
 'บ้านเลขที่',
 'หมู่',
 'หมู่บ้าน/ชุมชน',
 'ตำบล/แขวง',
 'เขต/อำเภอ/เทศบาล',
 'จังหวัด',
 'เบอร์โทรศัพท์',
 'ระดับการศึกษา',
 'การประกอบอาชีพ',
 'อาชีพหลัก',
 'โรคประจำตัว',
 'ลักษณะความพิการ',
 'สวัสดิการด้านเด็กและเยาวชน',
 'สวัสดิการด้านคนพิการ',
 'สวัสดิการด้านสตรี',
 'สวัสดิการด้านผู้สูงอายุ',
 'สวัสดิการด้านคนไร้ที่พึ่ง',
 'สวัสดิการด้านอื่นๆ',
 'ปัญหาด้านที่อยู่อาศัย',
 'ปัญหาด้านสุขภาพอนามัย',
 'ปัญหาด้านการศึกษา',
 'ปัญหาด้านการมีงานทำและมีรายได้',
 'ปัญหาด้านครอบครัว',
 'ปัญหาด้านความรุนแรงในครอบครัว/สังคม',
 'ปัญหาด้านการเป็นผู้เสียหายจากการค้ามนุษย์',
 'ปัญหาด้านการเข้าไม่ถึงสิทธิและความเป็นธรรมในสังคม',
 'ปัญหาด้านสภาพปัญหาสังคมอื่นๆ',
 'ประเภทกลุ่มเป้าหมาย',
 'รหัส cm',
 'ชื่อ นามสกุล cm',
 'หน่วยงานข

การทดลองอ่านด้วย `nrows=5` ช่วยให้เราเห็น

- จำนวนคอลัมน์
- ชื่อคอลัมน์
- ตัวอย่างค่า
- รูปแบบวันที่
- รหัสที่อาจต้องเก็บเป็นข้อความ

โดยไม่ต้องอ่านไฟล์ทั้งหมด

## 4. การกำหนด `encoding`

`encoding` คือวิธีที่ไฟล์ใช้แทนตัวอักษรเป็นข้อมูลดิจิทัล

Encoding ที่พบบ่อยในไฟล์ภาษาไทย ได้แก่

- `"utf-8"`
- `"utf-8-sig"`
- `"cp874"`

ควรใช้ Encoding ที่ตรงกับไฟล์ต้นทาง ไม่ควรเลือกโดยการเดาเมื่อมี Metadata หรือเอกสารจากเจ้าของข้อมูล

In [7]:
mso_preview = pd.read_csv(
    csv_file,
    encoding="utf-8-sig",
    nrows=5,
)

mso_preview


,วันที่แก้ไขข้อมูลล่าสุด,ชื่อสสว.,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,ลักษณะที่อยู่อาศัย,เลขบัตรประชาชน,เลขที่สมาชิกครัวเรือน,คำนำหน้า,ชื่อ,...,ปัญหาด้านการเป็นผู้เสียหายจากการค้ามนุษย์,ปัญหาด้านการเข้าไม่ถึงสิทธิและความเป็นธรรมในสังคม,ปัญหาด้านสภาพปัญหาสังคมอื่นๆ,ประเภทกลุ่มเป้าหมาย,รหัส cm,ชื่อ นามสกุล cm,หน่วยงานของ cm,ละติจูด,ลองจิจูด,สถานะการมีชีวิต
0,20230619 13:40:56.585,5,648ff7d6b64d06b6b0dc7f02,NaN,20230619 13:38:14.881,ที่ดินมีกรรมสิทธิ์ของตนเอง,5962222985aa759afbc332b96975a4c4d74a5f82a1ab8a...,648ff83350573a1fb581f155,เด็กหญิง,****,...,NaN,NaN,NaN,เด็กเล็ก,cm410011,****,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...,****,****,1
1,20220717 18:56:10.147,3,62d3f7a6d6f101550054198b,7.004022e+10,20220717 18:51:02.889,ที่ดินของผู้อื่น,62252665a08c86140a798453916f866d01ff17b8683341...,62d3f8aa24e6c5bf7e6edfdf,นางสาว,****,...,NaN,NaN,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm700018,****,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี,****,****,1
2,20240125 13:34:46.941,5,62a05c9df5674ecdd3805787,NaN,20220806 15:23:57.850,ที่ดินมีกรรมสิทธิ์ของตนเอง,761904ae1beb77d5de38fcbf34e757020c89291e467225...,62a05c9d63390df4b203ffdc,นางสาว,****,...,NaN,NaN,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm430010,****,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...,****,****,1
3,20240322 14:12:39.290,6,62908e378fa67bf5ad7d5555,NaN,20220527 15:39:19.197,ที่ดินมีกรรมสิทธิ์ของตนเอง,a94f022ed6a769f28d8cd2eefafed33d4185008de20a0b...,62908e3763390df4b20378f6,เด็กหญิง,****,...,NaN,ไม่สามารถเข้าถึงบริการของรัฐ|,NaN,เด็กเล็ก,cm340009,****,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...,****,****,1
4,20230315 09:27:11.763,4,6319b26ca3a37508384e36fc,NaN,20220809 16:14:20.652,ที่ดินมีกรรมสิทธิ์ของตนเอง,704921c77c739b12a7551cadae9998e06e50848647c058...,64112cc9dafc21d42cd4cfac,เด็กชาย,****,...,NaN,ไม่สามารถเข้าถึงบริการของรัฐ|,NaN,เด็กเล็ก,cm300020,****,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา,****,****,1


`utf-8-sig` เหมาะกับไฟล์ UTF-8 ที่อาจมี Byte Order Mark ซึ่งพบได้ในไฟล์ที่ส่งออกจากโปรแกรม Spreadsheet บางประเภท

## 5. การอ่านบางแถวด้วย `nrows`

`nrows` กำหนดจำนวนแถวข้อมูลที่ต้องการอ่าน

เหมาะสำหรับ

- ทดลองเปิดไฟล์ขนาดใหญ่
- ตรวจสอบโครงสร้าง
- ตรวจสอบชื่อคอลัมน์
- ทดลอง parameter
- ตรวจสอบข้อมูลก่อนอ่านทั้งไฟล์

In [8]:
mso_preview = pd.read_csv(
    csv_file,
    encoding="utf-8-sig",
    nrows=10,
)

mso_preview

,วันที่แก้ไขข้อมูลล่าสุด,ชื่อสสว.,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,ลักษณะที่อยู่อาศัย,เลขบัตรประชาชน,เลขที่สมาชิกครัวเรือน,คำนำหน้า,ชื่อ,...,ปัญหาด้านการเป็นผู้เสียหายจากการค้ามนุษย์,ปัญหาด้านการเข้าไม่ถึงสิทธิและความเป็นธรรมในสังคม,ปัญหาด้านสภาพปัญหาสังคมอื่นๆ,ประเภทกลุ่มเป้าหมาย,รหัส cm,ชื่อ นามสกุล cm,หน่วยงานของ cm,ละติจูด,ลองจิจูด,สถานะการมีชีวิต
0,20230619 13:40:56.585,5,648ff7d6b64d06b6b0dc7f02,NaN,20230619 13:38:14.881,ที่ดินมีกรรมสิทธิ์ของตนเอง,5962222985aa759afbc332b96975a4c4d74a5f82a1ab8a...,648ff83350573a1fb581f155,เด็กหญิง,****,...,NaN,NaN,NaN,เด็กเล็ก,cm410011,****,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...,****,****,1.0
1,20220717 18:56:10.147,3,62d3f7a6d6f101550054198b,7.004022e+10,20220717 18:51:02.889,ที่ดินของผู้อื่น,62252665a08c86140a798453916f866d01ff17b8683341...,62d3f8aa24e6c5bf7e6edfdf,นางสาว,****,...,NaN,NaN,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm700018,****,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี,****,****,1.0
2,20240125 13:34:46.941,5,62a05c9df5674ecdd3805787,NaN,20220806 15:23:57.850,ที่ดินมีกรรมสิทธิ์ของตนเอง,761904ae1beb77d5de38fcbf34e757020c89291e467225...,62a05c9d63390df4b203ffdc,นางสาว,****,...,NaN,NaN,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm430010,****,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...,****,****,1.0
3,20240322 14:12:39.290,6,62908e378fa67bf5ad7d5555,NaN,20220527 15:39:19.197,ที่ดินมีกรรมสิทธิ์ของตนเอง,a94f022ed6a769f28d8cd2eefafed33d4185008de20a0b...,62908e3763390df4b20378f6,เด็กหญิง,****,...,NaN,ไม่สามารถเข้าถึงบริการของรัฐ|,NaN,เด็กเล็ก,cm340009,****,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...,****,****,1.0
4,20230315 09:27:11.763,4,6319b26ca3a37508384e36fc,NaN,20220809 16:14:20.652,ที่ดินมีกรรมสิทธิ์ของตนเอง,704921c77c739b12a7551cadae9998e06e50848647c058...,64112cc9dafc21d42cd4cfac,เด็กชาย,****,...,NaN,ไม่สามารถเข้าถึงบริการของรัฐ|,NaN,เด็กเล็ก,cm300020,****,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา,****,****,1.0
5,20230316 13:23:07.617,3,63ad11f0049951d53b429c13,NaN,20221229 11:05:04.736,ที่ดินมีกรรมสิทธิ์ของตนเอง,ae202ff9b3617cdd20298d5ec6034324471091a03e4e1d...,63ad11f090998a158fd025cb,นาย,****,...,NaN,NaN,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm770005,****,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...,****,****,1.0
6,20221222 10:35:26.057,5,63a3d037cdf0fb5ddc8ba1f9,NaN,20221222 10:34:15.185,ที่ดินมีกรรมสิทธิ์ของตนเอง,84aec216e1f5cc877f10611de322ef60088ec6804de37b...,63a3d07186f724f8933ae232,เด็กหญิง,****,...,NaN,NaN,NaN,เด็กเล็ก,cm410006,****,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...,****,****,1.0
7,20230102 11:34:02.089,10,63d9eb642617b2b7aa05ea6c,NaN,20230102 11:32:36.354,ที่ดินมีกรรมสิทธิ์ของตนเอง,da9be1f726fa548f8794e2d4f3557782d00163c90b96ea...,63d9eb644714384a07f73ca5,นาง,****,...,NaN,NaN,NaN,ผู้สูงอายุ,cm800045,****,ศูนย์คุ้มครองคนไร้ที่พึ่ง จ.นครศรีธรรมราช,****,****,NaN
8,20220922 14:23:42.121,2,632c086a720e4eb0fded2524,2.512021e+10,20220922 14:02:02.435,เช่า,25c8391224689edfa8c14017d80fc2ce50fc576bc6f7a5...,632c0cd6963b3bd98cc1a495,นาง,****,...,NaN,NaN,NaN,ผู้สูงอายุ,cm270003,****,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...,****,****,1.0
9,20221223 10:02:55.712,4,63a519a7cdf0fb5ddc8bb44a,NaN,20221223 09:59:51.518,ที่ดินมีกรรมสิทธิ์ของตนเอง,cec2c54c7b5b8e8d86f21b6c4a80964fe30a77163e95f7...,63a51a395dee4ee3d2b4a990,เด็กหญิง,****,...,NaN,NaN,NaN,เด็ก,cm310013,****,บ้านพักเด็กและครอบครัวจังหวัดบุรีรัมย์,****,****,1.0


`nrows=10` หมายถึงอ่านข้อมูล 10 แถว ไม่รวม Header

## 6. การเลือกคอลัมน์ด้วย `usecols`

ไฟล์จริงอาจมีคอลัมน์จำนวนมาก แต่การวิเคราะห์หนึ่งงานอาจใช้เพียงบางคอลัมน์

`usecols` ช่วยให้ pandas อ่านเฉพาะคอลัมน์ที่ต้องการ

ข้อดี ได้แก่

- ลดการใช้หน่วยความจำ
- ลดเวลาในการอ่านไฟล์
- ทำให้ DataFrame กระชับ
- ทำให้ขอบเขตข้อมูลชัดเจน

In [9]:
mso_usecols = [
    "รหัสครัวเรือน",
    "วันที่สร้างครัวเรือน",
    "จังหวัด",
    "อายุ",
    "เพศ",
    "ระดับการศึกษา",
    "อาชีพหลัก",
    "ประเภทกลุ่มเป้าหมาย",
]

In [10]:
mso_selected_df = pd.read_csv(
    csv_file,
    encoding="utf-8-sig",
    usecols=mso_usecols,
    nrows=10,
)

mso_selected_df

,รหัสครัวเรือน,วันที่สร้างครัวเรือน,อายุ,เพศ,จังหวัด,ระดับการศึกษา,อาชีพหลัก,ประเภทกลุ่มเป้าหมาย
0,648ff7d6b64d06b6b0dc7f02,20230619 13:38:14.881,3,หญิง,อุดรธานี,ไม่ได้เรียนหนังสือ,เกษตรกรรม (พืช ปศุสัตว์ ประมง),เด็กเล็ก
1,62d3f7a6d6f101550054198b,20220717 18:51:02.889,55,หญิง,ราชบุรี,ไม่ได้เรียนหนังสือ,NaN,วัยผู้ใหญ่/วัยแรงงาน
2,62a05c9df5674ecdd3805787,20220806 15:23:57.850,33,หญิง,หนองคาย,มัธยมศึกษาตอนต้น,NaN,วัยผู้ใหญ่/วัยแรงงาน
3,62908e378fa67bf5ad7d5555,20220527 15:39:19.197,5,หญิง,อุบลราชธานี,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก
4,6319b26ca3a37508384e36fc,20220809 16:14:20.652,3,ชาย,นครราชสีมา,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก
5,63ad11f0049951d53b429c13,20221229 11:05:04.736,39,ชาย,ประจวบคีรีขันธ์,มัธยมศึกษาตอนต้น,ธุรกิจส่วนตัว/ค้าขาย/เจ้าของกิจการ,วัยผู้ใหญ่/วัยแรงงาน
6,63a3d037cdf0fb5ddc8ba1f9,20221222 10:34:15.185,4,หญิง,อุดรธานี,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก
7,63d9eb642617b2b7aa05ea6c,20230102 11:32:36.354,68,NaN,นครศรีธรรมราช,NaN,ธุรกิจส่วนตัว/ค้าขาย/เจ้าของกิจการ,ผู้สูงอายุ
8,632c086a720e4eb0fded2524,20220922 14:02:02.435,90,หญิง,สระแก้ว,ประถมศึกษา,NaN,ผู้สูงอายุ
9,63a519a7cdf0fb5ddc8bb44a,20221223 09:59:51.518,6,หญิง,บุรีรัมย์,ไม่ได้เรียนหนังสือ,NaN,เด็ก


หากชื่อใดใน `usecols` ไม่มีอยู่ในไฟล์ pandas จะเกิด `ValueError`

ดังนั้น ควรตรวจสอบชื่อจริงจาก

```python
mso_preview.columns.tolist()
```

ก่อนกำหนด `usecols`

## 7. การกำหนดชนิดข้อมูลด้วย `dtype`

บางคอลัมน์ประกอบด้วยตัวเลข แต่ไม่ได้มีความหมายเป็นจำนวนสำหรับการคำนวณ เช่น

- รหัสครัวเรือน
- รหัสประจำบ้าน
- รหัสเจ้าหน้าที่
- รหัสจังหวัด
- หมายเลขโทรศัพท์

ข้อมูลประเภทนี้ควรเก็บเป็นข้อความ เพื่อ

- รักษาเลขศูนย์นำหน้า
- ป้องกันการคำนวณโดยไม่ตั้งใจ
- รักษารูปแบบของรหัส

In [11]:
mso_dtype = {
    "รหัสครัวเรือน": "string",
    "รหัสประจำบ้าน": "string",
    "รหัส cm": "string",
}

In [12]:
mso_code_preview = pd.read_csv(
    csv_file,
    encoding="utf-8-sig",
    dtype=mso_dtype,
    nrows=10,
)

mso_code_preview[
    [
        "รหัสครัวเรือน",
        "รหัสประจำบ้าน",
        "รหัส cm",
    ]
].dtypes

รหัสครัวเรือน    string
รหัสประจำบ้าน    string
รหัส cm          string
dtype: object

## 8. การอ่านวันที่ด้วย `parse_dates`

คอลัมน์วันที่ที่อ่านจาก CSV มักถูกตีความเป็นข้อความ

`parse_dates` ใช้ระบุคอลัมน์ที่ต้องการให้ pandas พยายามแปลงเป็นชนิดวันที่และเวลา

In [13]:
mso_date_columns = [
    "วันที่แก้ไขข้อมูลล่าสุด",
    "วันที่สร้างครัวเรือน",
]

In [14]:
mso_df = pd.read_csv(
    csv_file,
    encoding="utf-8-sig",
    dtype=mso_dtype,
    parse_dates=mso_date_columns,
)

mso_df.head()

,วันที่แก้ไขข้อมูลล่าสุด,ชื่อสสว.,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,ลักษณะที่อยู่อาศัย,เลขบัตรประชาชน,เลขที่สมาชิกครัวเรือน,คำนำหน้า,ชื่อ,...,ปัญหาด้านการเป็นผู้เสียหายจากการค้ามนุษย์,ปัญหาด้านการเข้าไม่ถึงสิทธิและความเป็นธรรมในสังคม,ปัญหาด้านสภาพปัญหาสังคมอื่นๆ,ประเภทกลุ่มเป้าหมาย,รหัส cm,ชื่อ นามสกุล cm,หน่วยงานของ cm,ละติจูด,ลองจิจูด,สถานะการมีชีวิต
0,2023-06-19 13:40:56.585,5,648ff7d6b64d06b6b0dc7f02,<NA>,2023-06-19 13:38:14.881,ที่ดินมีกรรมสิทธิ์ของตนเอง,5962222985aa759afbc332b96975a4c4d74a5f82a1ab8a...,648ff83350573a1fb581f155,เด็กหญิง,****,...,NaN,NaN,NaN,เด็กเล็ก,cm410011,****,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...,****,****,1.0
1,2022-07-17 18:56:10.147,3,62d3f7a6d6f101550054198b,70040223124,2022-07-17 18:51:02.889,ที่ดินของผู้อื่น,62252665a08c86140a798453916f866d01ff17b8683341...,62d3f8aa24e6c5bf7e6edfdf,นางสาว,****,...,NaN,NaN,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm700018,****,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี,****,****,1.0
2,2024-01-25 13:34:46.941,5,62a05c9df5674ecdd3805787,<NA>,2022-08-06 15:23:57.850,ที่ดินมีกรรมสิทธิ์ของตนเอง,761904ae1beb77d5de38fcbf34e757020c89291e467225...,62a05c9d63390df4b203ffdc,นางสาว,****,...,NaN,NaN,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm430010,****,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...,****,****,1.0
3,2024-03-22 14:12:39.290,6,62908e378fa67bf5ad7d5555,<NA>,2022-05-27 15:39:19.197,ที่ดินมีกรรมสิทธิ์ของตนเอง,a94f022ed6a769f28d8cd2eefafed33d4185008de20a0b...,62908e3763390df4b20378f6,เด็กหญิง,****,...,NaN,ไม่สามารถเข้าถึงบริการของรัฐ|,NaN,เด็กเล็ก,cm340009,****,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...,****,****,1.0
4,2023-03-15 09:27:11.763,4,6319b26ca3a37508384e36fc,<NA>,2022-08-09 16:14:20.652,ที่ดินมีกรรมสิทธิ์ของตนเอง,704921c77c739b12a7551cadae9998e06e50848647c058...,64112cc9dafc21d42cd4cfac,เด็กชาย,****,...,NaN,ไม่สามารถเข้าถึงบริการของรัฐ|,NaN,เด็กเล็ก,cm300020,****,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา,****,****,1.0


In [15]:
mso_df[
    mso_date_columns
].dtypes

วันที่แก้ไขข้อมูลล่าสุด    datetime64[us]
วันที่สร้างครัวเรือน       datetime64[us]
dtype: object

`parse_dates` เป็นการขอให้ pandas พยายามอ่านวันที่

หากข้อมูลมี

- หลายรูปแบบในคอลัมน์เดียว
- วันที่ที่ไม่ถูกต้อง
- ปีหรือรูปแบบเฉพาะ
- ข้อความปะปน

บางค่าอาจยังไม่ถูกแปลงตามที่คาดหวัง จึงต้องตรวจสอบ `dtype` และตัวอย่างค่าหลังนำเข้าเสมอ

## 9. การอ่านไฟล์ Excel

คำสั่งหลักสำหรับอ่าน Excel คือ

```python
pd.read_excel()
```

การอ่านไฟล์ `.xlsx` ต้องใช้ Engine ที่รองรับ เช่น `openpyxl`

หาก Environment ยังไม่มี Package ที่จำเป็น ให้ติดตั้งก่อนเริ่มบทเรียน ไม่ควรวางคำสั่งติดตั้งไว้ในลำดับ `Run All` ของ Notebook หลัก

In [16]:
%pip install openpyxl


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
farmer_first_sheet_df = pd.read_excel(
    excel_file
)

farmer_first_sheet_df.head()

,department_code,department,pid,gender,birthdate,province_code,amphur_code,tambon_code,province,amphur,...,alro_allocated_area_ngan,alro_allocated_area_wa,is_alro,alro_postcode,updated_at,cpd_year_data,doae_reg_year,doae_corporation_be,doae_member_flag,doae_kaset_org_name
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2023,NaN,NaN,NaN,NaN
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2023,NaN,NaN,NaN,NaN
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2023,NaN,NaN,NaN,NaN
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2023,NaN,NaN,NaN,NaN
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2023,NaN,NaN,NaN,NaN


หากไม่กำหนด `sheet_name` pandas จะอ่าน Worksheet แรกโดยค่าเริ่มต้น

แต่ในงานจริงไม่ควรพึ่งลำดับ Worksheet โดยไม่ตรวจสอบ เพราะ

- ลำดับ Sheet อาจเปลี่ยน
- อาจมี Sheet ใหม่เพิ่มเข้ามา
- Sheet แรกอาจไม่ใช่ข้อมูลที่ต้องการ

## 10. ตรวจสอบและเลือก Worksheet

ใช้ `pd.ExcelFile()` เพื่อตรวจสอบโครงสร้างของ Workbook ก่อนอ่านข้อมูล

In [18]:
excel_data = pd.ExcelFile(
    excel_file
)

sheet_names = excel_data.sheet_names

sheet_names

['v_cpd_fragile',
 'v_dld_fragile',
 'v_doae_fragile',
 'v_dof_fragile',
 'v_raot_fragile']

เมื่อทราบชื่อ Worksheet แล้ว สามารถเลือกอ่านด้วย `sheet_name`

In [19]:
doae_df = pd.read_excel(
    excel_file,
    sheet_name="v_doae_fragile",
)

doae_df.head()

,department_code,department,pid,gender,birthdate,province_code,amphur_code,tambon_code,province,amphur,...,alro_allocated_area_ngan,alro_allocated_area_wa,is_alro,alro_postcode,updated_at,cpd_year_data,doae_reg_year,doae_corporation_be,doae_member_flag,doae_kaset_org_name
0,doae,กรมส่งเสริมการเกษตร,da98e3740c58278c3d0a4f01b23109a36b4a5a42a5fa72...,NaN,NaN,53.0,3.0,4.0,NaN,NaN,...,NaN,NaN,NaN,NaN,20250825,NaN,2023.0,NaN,สมาชิก,อื่นๆ
1,doae,กรมส่งเสริมการเกษตร,5295ea9f23f585562a5af56f1f3a6d12c463140aa8b367...,NaN,NaN,95.0,8.0,4.0,NaN,NaN,...,NaN,NaN,NaN,NaN,20250825,NaN,2015.0,NaN,สมาชิก,NaN
2,doae,กรมส่งเสริมการเกษตร,666e131e16b53c38ae8ee41917b2ab5ddd4f0e8674b19f...,NaN,NaN,93.0,7.0,3.0,NaN,NaN,...,NaN,NaN,NaN,NaN,20250825,NaN,2015.0,1.0,หัวหน้าครัวเรือน,NaN
3,doae,กรมส่งเสริมการเกษตร,bbd58a88987d6f79a3733dfd4885f0b74028004379ded3...,NaN,NaN,86.0,5.0,2.0,NaN,NaN,...,NaN,NaN,NaN,NaN,20250825,NaN,2015.0,NaN,สมาชิก,NaN
4,doae,กรมส่งเสริมการเกษตร,ebdb71900fee10f50d9f5b638e8a64d1fe3b7786fb69b7...,NaN,NaN,30.0,11.0,6.0,NaN,NaN,...,NaN,NaN,NaN,NaN,20250825,NaN,2016.0,NaN,สมาชิก,NaN


### ใช้ `usecols` และ `dtype` กับ Excel

`pd.read_excel()` รองรับ `usecols` และ `dtype` เช่นเดียวกับ `pd.read_csv()`

In [ ]:
farmer_usecols = [
    "department_code",
    "department",
    "pid",
    "province_code",
    "province",
    "amphur",
    "tambon",
    "is_farmer",
    "farmer_type",
    "main_occupation",
    "income_in",
    "income_out",
    "debts_in",
    "debts_out",
    "updated_at",
]

farmer_dtype = {
    "department_code": "string",
    "pid": "string",
    "province_code": "string",
}

In [21]:
farmer_selected_df = pd.read_excel(
    excel_file,
    sheet_name="v_doae_fragile",
    usecols=farmer_usecols,
    dtype=farmer_dtype,
)

farmer_selected_df.head()

,department_code,department,pid,province_code,province,amphur,tambon,is_farmer,farmer_type,main_occupation,income_in,income_out,debts_in,debts_out,updated_at
0,doae,กรมส่งเสริมการเกษตร,da98e3740c58278c3d0a4f01b23109a36b4a5a42a5fa72...,53,NaN,NaN,NaN,1,เกษตรกรด้านพืช,NaN,0.0,0.0,0.0,0.0,20250825
1,doae,กรมส่งเสริมการเกษตร,5295ea9f23f585562a5af56f1f3a6d12c463140aa8b367...,95,NaN,NaN,NaN,1,เกษตรกรด้านพืช,NaN,0.0,35000.0,0.0,0.0,20250825
2,doae,กรมส่งเสริมการเกษตร,666e131e16b53c38ae8ee41917b2ab5ddd4f0e8674b19f...,93,NaN,NaN,NaN,1,เกษตรกรด้านพืช,ประกอบการเกษตร,90000.0,20000.0,0.0,20000.0,20250825
3,doae,กรมส่งเสริมการเกษตร,bbd58a88987d6f79a3733dfd4885f0b74028004379ded3...,86,NaN,NaN,NaN,1,เกษตรกรด้านพืช,NaN,90000.0,10000.0,0.0,0.0,20250825
4,doae,กรมส่งเสริมการเกษตร,ebdb71900fee10f50d9f5b638e8a64d1fe3b7786fb69b7...,30,NaN,NaN,NaN,1,เกษตรกรด้านพืช,NaN,20000.0,60000.0,30000.0,250000.0,20250825


หากบาง Worksheet ไม่มีคอลัมน์ที่ระบุใน `usecols` จะเกิด Error

ก่อนใช้ชุดคอลัมน์เดียวกันกับหลาย Worksheet จึงควรตรวจสอบ Schema ของแต่ละ Sheet

## 11. การอ่านหลาย Worksheet

สามารถส่ง List ของชื่อ Worksheet ให้ `sheet_name`

In [22]:
selected_sheets = [
    "v_cpd_fragile",
    "v_dld_fragile",
    "v_doae_fragile",
]

In [23]:
farmer_sheets = pd.read_excel(
    excel_file,
    sheet_name=selected_sheets,
)

type(farmer_sheets)

dict

เมื่ออ่านหลาย Worksheet ผลลัพธ์เป็น Dictionary โดย

- key คือชื่อ Worksheet
- value คือ DataFrame ของ Worksheet นั้น

In [24]:
farmer_sheets.keys()

dict_keys(['v_cpd_fragile', 'v_dld_fragile', 'v_doae_fragile'])

In [25]:
farmer_sheets[
    "v_doae_fragile"
].head()

,department_code,department,pid,gender,birthdate,province_code,amphur_code,tambon_code,province,amphur,...,alro_allocated_area_ngan,alro_allocated_area_wa,is_alro,alro_postcode,updated_at,cpd_year_data,doae_reg_year,doae_corporation_be,doae_member_flag,doae_kaset_org_name
0,doae,กรมส่งเสริมการเกษตร,da98e3740c58278c3d0a4f01b23109a36b4a5a42a5fa72...,NaN,NaN,53.0,3.0,4.0,NaN,NaN,...,NaN,NaN,NaN,NaN,20250825,NaN,2023.0,NaN,สมาชิก,อื่นๆ
1,doae,กรมส่งเสริมการเกษตร,5295ea9f23f585562a5af56f1f3a6d12c463140aa8b367...,NaN,NaN,95.0,8.0,4.0,NaN,NaN,...,NaN,NaN,NaN,NaN,20250825,NaN,2015.0,NaN,สมาชิก,NaN
2,doae,กรมส่งเสริมการเกษตร,666e131e16b53c38ae8ee41917b2ab5ddd4f0e8674b19f...,NaN,NaN,93.0,7.0,3.0,NaN,NaN,...,NaN,NaN,NaN,NaN,20250825,NaN,2015.0,1.0,หัวหน้าครัวเรือน,NaN
3,doae,กรมส่งเสริมการเกษตร,bbd58a88987d6f79a3733dfd4885f0b74028004379ded3...,NaN,NaN,86.0,5.0,2.0,NaN,NaN,...,NaN,NaN,NaN,NaN,20250825,NaN,2015.0,NaN,สมาชิก,NaN
4,doae,กรมส่งเสริมการเกษตร,ebdb71900fee10f50d9f5b638e8a64d1fe3b7786fb69b7...,NaN,NaN,30.0,11.0,6.0,NaN,NaN,...,NaN,NaN,NaN,NaN,20250825,NaN,2016.0,NaN,สมาชิก,NaN


สามารถอ่านทุก Worksheet ด้วย

```python
sheet_name=None
```

ผลลัพธ์ยังคงเป็น Dictionary ของ DataFrame

In [26]:
all_farmer_sheets = pd.read_excel(
    excel_file,
    sheet_name=None,
)

all_farmer_sheets.keys()

dict_keys(['v_cpd_fragile', 'v_dld_fragile', 'v_doae_fragile', 'v_dof_fragile', 'v_raot_fragile'])

การใช้ `sheet_name=None` สะดวก แต่ควรระวัง Workbook ที่มี

- Sheet อธิบายข้อมูล
- Sheet สรุป
- Sheet ที่ซ่อน
- Sheet ที่มี Schema ต่างกัน

ไม่ควรรวมทุก Sheet โดยอัตโนมัติหากยังไม่ตรวจสอบโครงสร้าง

## 12. การรวมข้อมูลจากหลาย Worksheet

`pd.concat()` ใช้รวม DataFrame หลายชุดตามแนวแถว

ก่อนรวมควรตรวจสอบว่าแต่ละ DataFrame มี Schema ที่สอดคล้องกัน

In [27]:
for sheet_name, sheet_df in (
    farmer_sheets.items()
):
    print(
        sheet_name,
        sheet_df.shape,
    )

v_cpd_fragile (15, 38)
v_dld_fragile (10, 38)
v_doae_fragile (70, 38)


### ตรวจสอบชื่อคอลัมน์ของแต่ละ Worksheet

In [28]:
for sheet_name, sheet_df in (
    farmer_sheets.items()
):
    print(sheet_name)
    print(sheet_df.columns.tolist())
    print("---")

v_cpd_fragile
['department_code', 'department', 'pid', 'gender', 'birthdate', 'province_code', 'amphur_code', 'tambon_code', 'province', 'amphur', 'tambon', 'income_per_year', 'income_per_month', 'is_farmer', 'farmer_type', 'main_occupation', 'secondary_occupation', 'income_in', 'income_out', 'debts_in', 'debts_out', 'debts_bank_in', 'debts_bank_out', 'registration_date', 'registration_modify_date', 'registration_approve_date', 'alro_allocated_provinces', 'alro_allocated_area_rai', 'alro_allocated_area_ngan', 'alro_allocated_area_wa', 'is_alro', 'alro_postcode', 'updated_at', 'cpd_year_data', 'doae_reg_year', 'doae_corporation_be', 'doae_member_flag', 'doae_kaset_org_name']
---
v_dld_fragile
['department_code', 'department', 'pid', 'gender', 'birthdate', 'province_code', 'amphur_code', 'tambon_code', 'province', 'amphur', 'tambon', 'income_per_year', 'income_per_month', 'is_farmer', 'farmer_type', 'main_occupation', 'secondary_occupation', 'income_in', 'income_out', 'debts_in', 'debts_

หาก Schema เหมือนหรือสอดคล้องกัน สามารถเพิ่มชื่อแหล่งที่มาแล้วรวมข้อมูลได้

In [29]:
combined_farmer_list = []

for sheet_name, sheet_df in (
    farmer_sheets.items()
):
    temp_df = sheet_df.copy()

    temp_df["source_sheet"] = (
        sheet_name
    )

    combined_farmer_list.append(
        temp_df
    )

In [30]:
combined_farmer_df = pd.concat(
    combined_farmer_list,
    ignore_index=True,
)

combined_farmer_df.head()

,department_code,department,pid,gender,birthdate,province_code,amphur_code,tambon_code,province,amphur,...,alro_allocated_area_wa,is_alro,alro_postcode,updated_at,cpd_year_data,doae_reg_year,doae_corporation_be,doae_member_flag,doae_kaset_org_name,source_sheet
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2023.0,NaN,NaN,NaN,NaN,v_cpd_fragile
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2023.0,NaN,NaN,NaN,NaN,v_cpd_fragile
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2023.0,NaN,NaN,NaN,NaN,v_cpd_fragile
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2023.0,NaN,NaN,NaN,NaN,v_cpd_fragile
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2023.0,NaN,NaN,NaN,NaN,v_cpd_fragile


`ignore_index=True` สร้าง index ใหม่ต่อเนื่องหลังรวมข้อมูล

การเพิ่ม `source_sheet` ช่วยให้

- ตรวจสอบแหล่งที่มา
- เปรียบเทียบข้อมูลรายหน่วยงาน
- ติดตามปัญหากลับไปยัง Worksheet ต้นทาง

In [31]:
combined_farmer_df[
    "source_sheet"
].value_counts()

source_sheet
v_doae_fragile    70
v_cpd_fragile     15
v_dld_fragile     10
Name: count, dtype: int64

หาก DataFrame มีคอลัมน์ไม่เหมือนกัน `pd.concat()` จะสร้างคอลัมน์ทั้งหมดที่พบ และเติมค่าว่างให้แถวที่ไม่มีคอลัมน์นั้น

ดังนั้น การรวมสำเร็จโดยไม่เกิด Error ไม่ได้หมายความว่า Schema สอดคล้องกัน

## 13. การอ่านไฟล์จาก URL

หาก URL ชี้ไปยังข้อมูลโดยตรง pandas สามารถรับ URL แทน Path ได้

ตัวอย่างเช่น

```python
pd.read_csv(
    "https://example.org/data.csv"
)
```

การอ่านจาก URL ต้องอาศัย

- Internet
- URL ที่เข้าถึงได้
- Server ต้นทางพร้อมให้บริการ
- รูปแบบข้อมูลไม่เปลี่ยนแปลง

In [33]:
combined_farmer_list = [] 

for sheet in sheet_names: 
    temp_df = pd.read_excel( 
        excel_file, 
        sheet_name=sheet, 
    ) 
    
    temp_df["source_sheet"] = sheet 
    
    combined_farmer_list.append(temp_df) 
    
combined_farmer_df = pd.concat(combined_farmer_list, ignore_index=True) 
    
combined_farmer_df

,department_code,department,pid,gender,birthdate,province_code,amphur_code,tambon_code,province,amphur,...,alro_allocated_area_wa,is_alro,alro_postcode,updated_at,cpd_year_data,doae_reg_year,doae_corporation_be,doae_member_flag,doae_kaset_org_name,source_sheet
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2023.0,NaN,NaN,NaN,NaN,v_cpd_fragile
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2023.0,NaN,NaN,NaN,NaN,v_cpd_fragile
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2023.0,NaN,NaN,NaN,NaN,v_cpd_fragile
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2023.0,NaN,NaN,NaN,NaN,v_cpd_fragile
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2023.0,NaN,NaN,NaN,NaN,v_cpd_fragile
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,dof,กรมประมง,22af9655c9a862dca3fb3eda963860ce94f0347dde4d47...,NaN,NaN,NaN,NaN,40000.0,NaN,NaN,...,NaN,NaN,NaN,20250920.0,NaN,NaN,NaN,NaN,NaN,v_dof_fragile
96,dof,กรมประมง,3df944c366d9424ce33cef005ac11f5ec5cb824c59c132...,NaN,NaN,NaN,NaN,11140.0,NaN,NaN,...,NaN,NaN,NaN,20250920.0,NaN,NaN,NaN,NaN,NaN,v_dof_fragile
97,raot,การยางแห่งประเทศไทย,26e3056811b80168fdc2c16d4cc8357770707e9aec077a...,NaN,20001113.0,NaN,NaN,NaN,สงขลา,บางกล่ำ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,v_raot_fragile
98,raot,การยางแห่งประเทศไทย,2ded3d28c285ecbab97dbf994dcd5ae3d7d863cbf011c2...,NaN,19671009.0,NaN,NaN,NaN,ยะลา,เมืองยะลา,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,v_raot_fragile


ใน Environment ที่ไม่มี Internet หรือถูกจำกัด Network คำสั่งอ่าน URL อาจไม่ทำงาน

สำหรับ Workflow ที่ต้องทำซ้ำ ควรพิจารณา

1. ดาวน์โหลดไฟล์ต้นทาง
2. บันทึกสำเนาไว้ในพื้นที่ควบคุม
3. เก็บ Source URL และเวลาที่ดาวน์โหลดเป็น Metadata
4. อ่านจากไฟล์ที่บันทึกไว้

แนวทางนี้ช่วยให้ตรวจสอบย้อนหลังและทำซ้ำได้ง่ายกว่า

## 14. การอ่านไฟล์ข้อความที่มีตัวคั่น

ไฟล์ `.txt` อาจมีโครงสร้างเป็นตารางได้ หากแต่ละคอลัมน์คั่นด้วยตัวอักษรที่ชัดเจน เช่น

- comma `,`
- tab `\t`
- semicolon `;`
- pipe `|`

`pd.read_csv()` สามารถอ่านไฟล์ลักษณะนี้ได้ด้วย parameter `sep`

ไฟล์ประชากรตัวอย่างใช้ `|` เป็นตัวคั่น จึงกำหนด

```python
sep="|"
```

ขั้นแรกควรทดลองอ่านเพียงบางแถว

In [32]:
population_preview = pd.read_csv(
    population_url,
    sep="|",
    nrows=5,
)

population_preview

,กรุงเทพมหานคร,11640,10838,14323,13274,15588,14635,15814,14567,17733,...,37804,28545,66349,1333,1150,2483,2526272,2896296,5422568,Unnamed: 220
0,เขตพระนคร,52,40,55,47,73,48,70,64,78,...,388,256,644,1,3,4,18723,19685,38408,NaN
1,แขวงพระบรมมหาราชวัง,4,2,7,2,3,4,2,5,4,...,17,16,33,1,0,1,1684,1104,2788,NaN
2,แขวงวังบูรพาภิรมย์,10,9,12,10,20,8,14,15,26,...,110,90,200,0,0,0,4172,4066,8238,NaN
3,แขวงวัดราชบพิธ,5,6,4,2,5,3,1,3,1,...,24,23,47,0,0,0,1218,1509,2727,NaN
4,แขวงสำราญราษฎร์,4,1,5,1,6,5,5,3,5,...,35,12,47,0,0,0,1332,1260,2592,NaN


ผลลัพธ์ผิดปกติ เพราะไฟล์ไม่มี Header แต่ pandas ใช้แถวแรกเป็นชื่อคอลัมน์โดยค่าเริ่มต้น

จึงต้องกำหนด

```python
header=None
```

In [33]:
population_preview = pd.read_csv(
    population_url,
    sep="|",
    header=None,
    nrows=5,
)

population_preview

,0,1,2,3,4,5,6,7,8,9,...,211,212,213,214,215,216,217,218,219,220
0,กรุงเทพมหานคร,11640,10838,14323,13274,15588,14635,15814,14567,17733,...,37804,28545,66349,1333,1150,2483,2526272,2896296,5422568,NaN
1,เขตพระนคร,52,40,55,47,73,48,70,64,78,...,388,256,644,1,3,4,18723,19685,38408,NaN
2,แขวงพระบรมมหาราชวัง,4,2,7,2,3,4,2,5,4,...,17,16,33,1,0,1,1684,1104,2788,NaN
3,แขวงวังบูรพาภิรมย์,10,9,12,10,20,8,14,15,26,...,110,90,200,0,0,0,4172,4066,8238,NaN
4,แขวงวัดราชบพิธ,5,6,4,2,5,3,1,3,1,...,24,23,47,0,0,0,1218,1509,2727,NaN


เมื่อใช้ `header=None` pandas จะสร้างชื่อคอลัมน์เป็นตัวเลข `0`, `1`, `2`, ... ชั่วคราว

ก่อนกำหนดชื่อจริง ควรตรวจสอบจำนวนคอลัมน์ที่อ่านได้

In [34]:
population_preview.shape

(5, 221)

ไฟล์ตัวอย่างมีตัวคั่น `|` ท้ายบรรทัด ทำให้ pandas อ่านคอลัมน์ว่างเพิ่มมาอีกหนึ่งคอลัมน์

จึงควรตรวจสอบคอลัมน์สุดท้าย

In [35]:
population_preview.iloc[
    :,
    -3:
]

,218,219,220
0,2896296,5422568,NaN
1,19685,38408,NaN
2,1104,2788,NaN
3,4066,8238,NaN
4,1509,2727,NaN


หากคอลัมน์สุดท้ายว่างทั้งหมด แสดงว่าเป็นผลจากตัวคั่นท้ายบรรทัด ไม่ใช่ตัวแปรข้อมูลจริง

## 15. การกำหนดชื่อคอลัมน์ให้ไฟล์ที่ไม่มี Header

จากโครงสร้างข้อมูลของกรมการปกครองตามลิงค์ต่อไปนี้

```text
https://stat.bora.dopa.go.th/new_stat/webPage/statByAge.php
```

ไฟล์จำนวนประชากรแยกรายอายุมี field หลัก ๆ ดังนี้

1. `DESC-CCAATT` รายละเอียดจังหวัด หรืออำเภอ หรือตำบล 
2. ชุดข้อมูลประชากรชายและหญิงรายอายุ ตั้งแต่อายุ 0 ถึง 101 โดยอายุ 0 หมายถึงอายุน้อยกว่า 1 ปี และอายุ 101 หมายถึงอายุมากกว่า 100 ปี 
3. ข้อมูลกลุ่มพิเศษและยอดรวม เช่น 
- ประชากรชาย/หญิงเกิดปีจันทรคติ 
- ประชากรชาย/หญิงที่มีชื่ออยู่ในทะเบียนบ้านกลาง 
- ประชากรชาย/หญิงที่มิใช่สัญชาติไทย 
- ประชากรชาย/หญิงที่อยู่ระหว่างการย้าย 
- ประชากรชายทั้งหมด 
- ประชากรหญิงทั้งหมด 
- ผลรวมประชากรทั้งหมด 

ดังนั้นเราจะต้องสร้างชื่อ column ให้ครบตามลำดับ field ของไฟล์ ดังนี้

1. คำอธิบายพื้นที่ 1 คอลัมน์
2. จำนวนประชากรชายและหญิงแยกตามอายุ 0–101 ปี
3. คอลัมน์สรุปท้ายไฟล์ 15 คอลัมน์

สามารถสร้างชื่อคอลัมน์อายุด้วย Loop

In [36]:
age_columns = []

for age in range(102):
    age_columns.append(
        f"male_age_{age}"
    )

    age_columns.append(
        f"female_age_{age}"
    )

In [37]:
len(age_columns)

204

มีอายุ 102 ค่า และแต่ละอายุมีชายกับหญิง

ดังนั้นจำนวนคอลัมน์อายุคือ

```text
102 × 2 = 204
```

In [38]:
summary_columns = [
    "male_tdob",
    "female_tdob",
    "total_tdob",
    "male_xhouse",
    "female_xhouse",
    "total_xhouse",
    "male_other_nat",
    "female_other_nat",
    "total_other_nat",
    "male_move",
    "female_move",
    "total_move",
    "male_tot",
    "female_tot",
    "total_tot",
]

In [39]:
population_columns = (
    ["desc_ccaatt"]
    + age_columns
    + summary_columns
)

len(population_columns)

220

จำนวนชื่อคอลัมน์ที่คาดหวังคือ

```text
1 + 204 + 15 = 220 คอลัมน์
```

แต่ไฟล์ที่ทดลองอ่านมี 221 คอลัมน์ เพราะมีคอลัมน์ว่างจากตัวคั่นท้ายบรรทัด

จึงต้องเลือกเฉพาะ 220 คอลัมน์ข้อมูลจริง

In [40]:
expected_column_count = len(
    population_columns
)

expected_column_count

220

In [41]:
population_df = pd.read_csv(
    population_url,
    sep="|",
    header=None,
    names=population_columns,
    usecols=range(
        expected_column_count
    ),
    dtype={
        "desc_ccaatt": "string",
    },
)

population_df.head()

,desc_ccaatt,male_age_0,female_age_0,male_age_1,female_age_1,male_age_2,female_age_2,male_age_3,female_age_3,male_age_4,...,total_xhouse,male_other_nat,female_other_nat,total_other_nat,male_move,female_move,total_move,male_tot,female_tot,total_tot
0,กรุงเทพมหานคร,11640,10838,14323,13274,15588,14635,15814,14567,17733,...,73564,37804,28545,66349,1333,1150,2483,2526272,2896296,5422568
1,เขตพระนคร,52,40,55,47,73,48,70,64,78,...,436,388,256,644,1,3,4,18723,19685,38408
2,แขวงพระบรมมหาราชวัง,4,2,7,2,3,4,2,5,4,...,75,17,16,33,1,0,1,1684,1104,2788
3,แขวงวังบูรพาภิรมย์,10,9,12,10,20,8,14,15,26,...,62,110,90,200,0,0,0,4172,4066,8238
4,แขวงวัดราชบพิธ,5,6,4,2,5,3,1,3,1,...,6,24,23,47,0,0,0,1218,1509,2727


การใช้

```python
usecols=range(expected_column_count)
```

ทำให้เลือกเฉพาะคอลัมน์ตำแหน่ง `0` ถึง `219` และไม่อ่านคอลัมน์ว่างที่เกิดจากตัวคั่นท้ายบรรทัด

In [42]:
population_df.shape

(231, 220)

In [43]:
population_df.columns.tolist()[:10]

['desc_ccaatt',
 'male_age_0',
 'female_age_0',
 'male_age_1',
 'female_age_1',
 'male_age_2',
 'female_age_2',
 'male_age_3',
 'female_age_3',
 'male_age_4']

In [44]:
population_df.columns.tolist()[-15:]

['male_tdob',
 'female_tdob',
 'total_tdob',
 'male_xhouse',
 'female_xhouse',
 'total_xhouse',
 'male_other_nat',
 'female_other_nat',
 'total_other_nat',
 'male_move',
 'female_move',
 'total_move',
 'male_tot',
 'female_tot',
 'total_tot']

## 16. ตรวจสอบความสอดคล้องของโครงสร้าง

ก่อนอ่านไฟล์จริงด้วย `names` ควรเปรียบเทียบ

- จำนวนคอลัมน์ที่อ่านได้จาก Preview
- จำนวนชื่อคอลัมน์ที่สร้าง
- จำนวนคอลัมน์ว่างจากตัวคั่นท้ายบรรทัด

In [45]:
observed_column_count = (
    population_preview.shape[1]
)

expected_column_count = len(
    population_columns
)

trailing_empty_column_count = (
    observed_column_count
    - expected_column_count
)

print(
    "Observed columns:",
    observed_column_count,
)

print(
    "Expected columns:",
    expected_column_count,
)

print(
    "Extra columns:",
    trailing_empty_column_count,
)

Observed columns: 221
Expected columns: 220
Extra columns: 1


หากจำนวนคอลัมน์ไม่เป็นไปตามที่คาด ไม่ควรบังคับชื่อคอลัมน์แล้วดำเนินงานต่อทันที

ควรตรวจสอบว่า

- โครงสร้างต้นทางเปลี่ยนหรือไม่
- ตัวคั่นถูกต้องหรือไม่
- มี Header หรือไม่
- มีตัวคั่นท้ายบรรทัดหรือไม่
- เอกสาร Data Dictionary ยังเป็นปัจจุบันหรือไม่

## 17. ข้อผิดพลาดที่พบบ่อย

### 17.1 ไม่พบไฟล์

หาก Path ไม่ถูกต้อง จะเกิด `FileNotFoundError`

In [46]:
# ตัวอย่างที่ทำให้เกิด
# FileNotFoundError

pd.read_csv(
    "file_not_found.csv"
)

FileNotFoundError: [Errno 2] No such file or directory: 'file_not_found.csv'

ควรตรวจสอบด้วย

```python
file_path.exists()
file_path.is_file()
```

ก่อนนำเข้า

### 17.2 ใช้ Encoding ไม่ตรงกับไฟล์

In [47]:
# ตัวอย่างที่อาจทำให้เกิด
# UnicodeDecodeError

pd.read_csv(
    csv_file,
    encoding="ascii",
)

UnicodeDecodeError: 'ascii' codec can't decode byte 0xe0 in position 0: ordinal not in range(128)

ควรใช้ Encoding ตามข้อมูลจากเจ้าของไฟล์ หรือทดลองจาก Encoding ที่เป็นไปได้อย่างมีหลักฐาน

### 17.3 ชื่อคอลัมน์ใน `usecols` ไม่ตรงกับไฟล์

In [48]:
# ตัวอย่างที่ทำให้เกิด
# ValueError

pd.read_csv(
    csv_file,
    usecols=[
        "column_not_found",
    ],
)

ValueError: Usecols do not match columns, columns expected but not found: ['column_not_found']

ควรตรวจสอบชื่อคอลัมน์ด้วย

```python
preview.columns.tolist()
```

ชื่อคอลัมน์อาจมี

- ช่องว่าง
- ตัวพิมพ์แตกต่างกัน
- อักขระพิเศษ
- ชื่อที่คล้ายกันแต่ไม่เหมือนกัน

### 17.4 ไม่กำหนด `dtype` ให้คอลัมน์รหัส

หากรหัสถูกอ่านเป็นตัวเลข เลขศูนย์นำหน้าอาจหายไป และไม่สามารถคืนรูปแบบเดิมได้อย่างน่าเชื่อถือ

### 17.5 คิดว่า `parse_dates` สำเร็จโดยไม่ตรวจสอบ

หลังนำเข้าควรตรวจสอบ

```python
dataframe[date_columns].dtypes
```

ไม่ควรสรุปว่าเป็นวันที่เพียงเพราะคำสั่งไม่เกิด Error

### 17.6 อ่าน Excel โดยไม่ระบุ Worksheet

การใช้ค่าเริ่มต้นอาจอ่าน Sheet ผิด หากลำดับใน Workbook เปลี่ยน

ควรตรวจสอบ `sheet_names` และระบุ `sheet_name` ให้ชัดเจน

### 17.7 รวมหลาย Worksheet โดยไม่ตรวจสอบ Schema

`pd.concat()` อาจรวมข้อมูลสำเร็จแม้คอลัมน์ไม่ตรงกัน โดยเติมค่าว่างในคอลัมน์ที่ขาด

จึงต้องตรวจสอบรายชื่อคอลัมน์ก่อนรวม

### 17.8 เข้าใจแถวแรกเป็น Header ทั้งที่ไฟล์ไม่มี Header

ให้ใช้

```python
header=None
```

แล้วกำหนดชื่อคอลัมน์ด้วย `names`

### 17.9 จำนวน `names` ไม่ตรงกับข้อมูลจริง

หากจำนวนชื่อคอลัมน์ไม่ตรงกับจำนวน Field อาจเกิด

- การเลื่อนตำแหน่งข้อมูล
- คอลัมน์แรกกลายเป็น index
- ข้อมูลท้ายแถวหาย
- Parser warning

ควรตรวจสอบจำนวนคอลัมน์จาก Preview ก่อนกำหนด `names`

### 17.10 อ่านจาก URL โดยไม่วางแผนกรณี Network ล้มเหลว

URL อาจเปลี่ยน ปิดให้บริการ หรือเข้าถึงไม่ได้

Workflow ที่ต้องการทำซ้ำควรพิจารณาเก็บสำเนาไฟล์ต้นทางและ Metadata ของการดาวน์โหลด

# แบบฝึกหัดท้ายบท

แบบฝึกหัดนี้จำลองสถานการณ์นำเข้าข้อมูลจริงจากหลายแหล่ง

แบบฝึกหัดครอบคลุม

- การอ่าน CSV ด้วยรูปแบบมาตรฐาน
- การสร้าง Function สำหรับนำเข้า CSV
- การอ่านและรวมหลาย Worksheet
- การสร้าง URL และ Schema แบบเปลี่ยน Parameter ได้
- การสร้าง Function กลางสำหรับเลือกวิธีนำเข้า

## แบบฝึกหัดที่ 1: เตรียมข้อมูล CSV สำหรับการวิเคราะห์

คุณได้รับไฟล์

```python
"msdhs_ops_mso_logbook.csv"
```

ให้ทำงานต่อไปนี้

1. กำหนด Path เป็นตัวแปร `mso_file`
2. ตรวจสอบว่าไฟล์มีอยู่จริง
3. กำหนด `mso_usecols` เป็นคอลัมน์ต่อไปนี้

```python
[
    "รหัสครัวเรือน",
    "รหัสประจำบ้าน",
    "วันที่สร้างครัวเรือน",
    "จังหวัด",
    "อายุ",
    "เพศ",
    "ระดับการศึกษา",
    "อาชีพหลัก",
    "ประเภทกลุ่มเป้าหมาย",
    "รหัส cm",
    "หน่วยงานของ cm",
]
```

4. กำหนด `mso_dtype` ให้คอลัมน์ต่อไปนี้เป็น `"string"`

```python
[
    "รหัสครัวเรือน",
    "รหัสประจำบ้าน",
    "รหัส cm",
]
```

5. อ่านไฟล์ด้วย `encoding="utf-8-sig"`
6. ใช้ `parse_dates` กับ `"วันที่สร้างครัวเรือน"`
7. เก็บผลลัพธ์ใน `mso_work_df`
8. ตรวจสอบ `.head()`, `.shape` และ `.dtypes`

In [24]:
# เขียนคำตอบของคุณใน Cell นี้

> ### เฉลยแบบฝึกหัดที่ 1
>
> ควรแยก Path, รายชื่อคอลัมน์ และ Dictionary ของชนิดข้อมูลออกจากคำสั่ง `pd.read_csv()` เพื่อให้ตรวจสอบและนำกลับมาใช้ซ้ำได้ง่าย

In [49]:
mso_file = Path(
    "msdhs_ops_mso_logbook.csv"
)

if not mso_file.is_file():
    raise FileNotFoundError(
        f"File not found: {mso_file}"
    )

mso_usecols = [
    "รหัสครัวเรือน",
    "รหัสประจำบ้าน",
    "วันที่สร้างครัวเรือน",
    "จังหวัด",
    "อายุ",
    "เพศ",
    "ระดับการศึกษา",
    "อาชีพหลัก",
    "ประเภทกลุ่มเป้าหมาย",
    "รหัส cm",
    "หน่วยงานของ cm",
]

mso_dtype = {
    "รหัสครัวเรือน": "string",
    "รหัสประจำบ้าน": "string",
    "รหัส cm": "string",
}

mso_work_df = pd.read_csv(
    mso_file,
    encoding="utf-8-sig",
    usecols=mso_usecols,
    dtype=mso_dtype,
    parse_dates=[
        "วันที่สร้างครัวเรือน",
    ],
)

In [50]:
print(mso_work_df.head())

print(mso_work_df.shape)

print(mso_work_df.dtypes)

              รหัสครัวเรือน รหัสประจำบ้าน    วันที่สร้างครัวเรือน  อายุ   เพศ  \
0  648ff7d6b64d06b6b0dc7f02          <NA> 2023-06-19 13:38:14.881     3  หญิง   
1  62d3f7a6d6f101550054198b   70040223124 2022-07-17 18:51:02.889    55  หญิง   
2  62a05c9df5674ecdd3805787          <NA> 2022-08-06 15:23:57.850    33  หญิง   
3  62908e378fa67bf5ad7d5555          <NA> 2022-05-27 15:39:19.197     5  หญิง   
4  6319b26ca3a37508384e36fc          <NA> 2022-08-09 16:14:20.652     3   ชาย   

       จังหวัด       ระดับการศึกษา                       อาชีพหลัก  \
0     อุดรธานี  ไม่ได้เรียนหนังสือ  เกษตรกรรม (พืช ปศุสัตว์ ประมง)   
1      ราชบุรี  ไม่ได้เรียนหนังสือ                             NaN   
2      หนองคาย    มัธยมศึกษาตอนต้น                             NaN   
3  อุบลราชธานี  ไม่ได้เรียนหนังสือ                             NaN   
4   นครราชสีมา  ไม่ได้เรียนหนังสือ                             NaN   

    ประเภทกลุ่มเป้าหมาย   รหัส cm  \
0              เด็กเล็ก  cm410011   
1  วัยผู้ใหญ่/วัยแ

## แบบฝึกหัดที่ 2: สร้าง Function สำหรับอ่าน CSV

สร้าง Function ชื่อ `read_mso_csv()`

```python
def read_mso_csv(
    file_path,
    selected_columns,
    code_columns,
    date_columns,
):
    ...
```

Function ต้อง

1. ตรวจสอบว่าไฟล์มีอยู่จริง
2. สร้าง `dtype_dict` จาก `code_columns`
3. อ่านไฟล์ด้วย `pd.read_csv()`
4. ใช้ `encoding="utf-8-sig"`
5. ใช้ `usecols=selected_columns`
6. ใช้ `dtype=dtype_dict`
7. ใช้ `parse_dates=date_columns`
8. คืน DataFrame ที่อ่านได้

In [25]:
# เขียนคำตอบของคุณใน Cell นี้

> ### เฉลยแบบฝึกหัดที่ 2
>
> ใช้ Dictionary comprehension หรือ Loop สร้างคู่ `"column_name": "string"` จากรายชื่อคอลัมน์รหัส
>
> Function ควรคืน DataFrame ด้วย `return` ไม่ควรมีเพียง `print()`

In [52]:
def read_mso_csv(
    file_path,
    selected_columns,
    code_columns,
    date_columns,
):
    file_path = Path(file_path)

    if not file_path.is_file():
        raise FileNotFoundError(
            f"File not found: {file_path}"
        )

    dtype_dict = {}

    for column in code_columns:
        dtype_dict[column] = "string"

    dataframe = pd.read_csv(
        file_path,
        encoding="utf-8-sig",
        usecols=selected_columns,
        dtype=dtype_dict,
        parse_dates=date_columns,
    )

    return dataframe

In [53]:
mso_from_function_df = read_mso_csv(
    file_path=mso_file,
    selected_columns=mso_usecols,
    code_columns=[
        "รหัสครัวเรือน",
        "รหัสประจำบ้าน",
        "รหัส cm",
    ],
    date_columns=[
        "วันที่สร้างครัวเรือน",
    ],
)

mso_from_function_df.head()

,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,อายุ,เพศ,จังหวัด,ระดับการศึกษา,อาชีพหลัก,ประเภทกลุ่มเป้าหมาย,รหัส cm,หน่วยงานของ cm
0,648ff7d6b64d06b6b0dc7f02,<NA>,2023-06-19 13:38:14.881,3,หญิง,อุดรธานี,ไม่ได้เรียนหนังสือ,เกษตรกรรม (พืช ปศุสัตว์ ประมง),เด็กเล็ก,cm410011,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
1,62d3f7a6d6f101550054198b,70040223124,2022-07-17 18:51:02.889,55,หญิง,ราชบุรี,ไม่ได้เรียนหนังสือ,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm700018,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี
2,62a05c9df5674ecdd3805787,<NA>,2022-08-06 15:23:57.850,33,หญิง,หนองคาย,มัธยมศึกษาตอนต้น,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm430010,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
3,62908e378fa67bf5ad7d5555,<NA>,2022-05-27 15:39:19.197,5,หญิง,อุบลราชธานี,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm340009,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
4,6319b26ca3a37508384e36fc,<NA>,2022-08-09 16:14:20.652,3,ชาย,นครราชสีมา,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm300020,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา


## แบบฝึกหัดที่ 3: อ่านและรวมข้อมูลจากหลาย Worksheet

คุณได้รับไฟล์ Excel ชื่อ

"moac_opsmoac_fragile_farmer.xlsx"

ไฟล์นี้มีข้อมูลจากหลายหน่วยงานอยู่ในหลาย sheet

ทีมวิเคราะห์ต้องการ function ที่สามารถอ่านหลาย sheet และรวมเป็น DataFrame เดียวได้

เพราะในอนาคตอาจต้องเลือก sheet ไม่เหมือนกันในแต่ละงาน

ให้สร้าง Function ชื่อ `read_and_combine_excel_sheets()`

```python
def read_and_combine_excel_sheets(
    file_path,
    sheet_list,
    selected_columns,
    dtype_dict,
):
    ...
```

Function ต้อง

1. ตรวจสอบว่าไฟล์มีอยู่จริง
2. ใช้ `pd.ExcelFile()` ตรวจสอบรายชื่อ Worksheet
3. ใช้ Loop ผ่าน `sheet_list`
4. หากไม่พบ Worksheet ให้แสดง  
   `"Sheet not found: <sheet name>"`
5. อ่าน Worksheet ที่พบด้วย `usecols` และ `dtype`
6. เพิ่มคอลัมน์ `source_sheet`
7. เก็บ DataFrame ไว้ใน List
8. รวมด้วย `pd.concat(ignore_index=True)`
9. หากอ่านไม่ได้เลย ให้คืน DataFrame ว่าง

ทดลองด้วย

```python
target_sheets = [
    "v_cpd_fragile",
    "v_dld_fragile",
    "v_doae_fragile",
]
```

In [26]:
# เขียนคำตอบของคุณใน Cell นี้

> ### เฉลยแบบฝึกหัดที่ 3
>
> ควรตรวจชื่อ Worksheet ก่อนอ่าน และใช้ `.copy()` ก่อนเพิ่ม `source_sheet`
>
> การตรวจ `if not dataframe_list` ช่วยรองรับกรณีไม่มี Worksheet ใดถูกอ่านสำเร็จ

In [54]:
def read_and_combine_excel_sheets(
    file_path,
    sheet_list,
    selected_columns,
    dtype_dict,
):
    file_path = Path(file_path)

    if not file_path.is_file():
        raise FileNotFoundError(
            f"File not found: {file_path}"
        )

    excel_data = pd.ExcelFile(
        file_path
    )

    available_sheets = (
        excel_data.sheet_names
    )

    dataframe_list = []

    for sheet_name in sheet_list:
        if (
            sheet_name
            not in available_sheets
        ):
            print(
                "Sheet not found:"
                f" {sheet_name}"
            )
            continue

        sheet_df = pd.read_excel(
            excel_data,
            sheet_name=sheet_name,
            usecols=selected_columns,
            dtype=dtype_dict,
        )

        sheet_df = sheet_df.copy()

        sheet_df["source_sheet"] = (
            sheet_name
        )

        dataframe_list.append(
            sheet_df
        )

    if not dataframe_list:
        return pd.DataFrame()

    combined_df = pd.concat(
        dataframe_list,
        ignore_index=True,
    )

    return combined_df

In [55]:
target_sheets = [
    "v_cpd_fragile",
    "v_dld_fragile",
    "v_doae_fragile",
]

farmer_usecols = [
    "department_code",
    "department",
    "pid",
    "province_code",
    "province",
    "amphur",
    "tambon",
    "is_farmer",
    "farmer_type",
    "main_occupation",
    "income_in",
    "income_out",
    "debts_in",
    "debts_out",
    "updated_at",
]

farmer_dtype = {
    "department_code": "string",
    "pid": "string",
    "province_code": "string",
}

In [56]:
combined_farmer_df = (
    read_and_combine_excel_sheets(
        file_path=excel_file,
        sheet_list=target_sheets,
        selected_columns=farmer_usecols,
        dtype_dict=farmer_dtype,
    )
)

combined_farmer_df.head()

,department_code,department,pid,province_code,province,amphur,tambon,is_farmer,farmer_type,main_occupation,income_in,income_out,debts_in,debts_out,updated_at,source_sheet
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile


In [57]:
print(combined_farmer_df.shape)

combined_farmer_df[
    "source_sheet"
].value_counts()

(95, 16)


source_sheet
v_doae_fragile    70
v_cpd_fragile     15
v_dld_fragile     10
Name: count, dtype: int64

## แบบฝึกหัดที่ 4: สร้าง function สำหรับนำเข้าข้อมูลประชากรรายอายุโดยเปลี่ยนปีและจังหวัดได้ 
คุณต้องนำเข้าข้อมูลจำนวนประชากรแยกรายอายุจากเว็บไซต์ของกรมการปกครอง

ไฟล์ข้อมูลมีรูปแบบ URL ดังนี้

"https://stat.bora.dopa.go.th/new_stat/file/{เลขท้าย 2 ตัวหลังของปี พศ}12/{เลขท้าย 2 ตัวหลังของปี พศ}12cc{รหัสจังหวัด}.txt"

ตัวอย่างเช่น ข้อมูลจังหวัดกรุงเทพมหานคร ปี พ.ศ. 2568 ใช้ URL

"https://stat.bora.dopa.go.th/new_stat/file/6812/6812cc10.txt"

โดย

68 คือเลขท้าย 2 ตัวของปี พ.ศ. 2568

10 คือรหัสจังหวัดกรุงเทพมหานคร

ในงานจริง นักวิเคราะห์อาจต้องเปลี่ยนปีหรือจังหวัดอยู่บ่อยครั้ง ดังนั้นไม่ควรเขียน URL แบบตายตัว 

แต่ควรสร้าง function ที่รับปีและรหัสจังหวัด แล้วสร้าง URL ให้อัตโนมัติ

ข้อกำหนด:

ให้สร้าง function ชื่อ `create_population_url()` โดยมี parameter ดังนี้

```python
def create_population_url(buddhist_year, province_code):
    ...
```

function นี้ต้องทำงานดังนี้
1. รับปี พ.ศ. จาก buddhist_year เช่น 2568
2. รับรหัสจังหวัดจาก province_code เช่น "10"
3. ดึงเลขท้าย 2 ตัวของปี พ.ศ. เช่น 2568 → "68"
4. สร้าง URL ตามรูปแบบ

"https://stat.bora.dopa.go.th/new_stat/file/{year_suffix}12/{year_suffix}12cc{province_code}.txt"

5. return URL ที่สร้างได้

จากนั้นให้สร้าง function ชื่อ `create_population_columns()`

```python
def create_population_columns(): ...
```

function นี้ต้องทำงานดังนี้
1. สร้าง list ชื่อ age_columns
2. ใช้ loop สร้าง column ตั้งแต่

male_age_0
female_age_0

ไปจนถึง

male_age_101
female_age_101

3. สร้าง list ของ column ท้ายไฟล์ชื่อ summary_columns

```python
[
    "male_tdob",
    "female_tdob",
    "total_tdob",
    "male_xhouse",
    "female_xhouse",
    "total_xhouse",
    "male_other_nat",
    "female_other_nat",
    "total_other_nat",
    "male_move",
    "female_move",
    "total_move",
    "male_tot",
    "female_tot",
    "total_tot"
]
```

4. return list ของ column ทั้งหมดในลำดับนี้

["desc_ccaatt"] + age_columns + summary_columns

จากนั้นให้สร้าง function ชื่อ `read_population_by_age()`

```python
def read_population_by_age(buddhist_year, province_code):
    ...
```

function นี้ต้องทำงานดังนี้
1. เรียกใช้ create_population_url() เพื่อสร้าง URL จากปีและรหัสจังหวัด
2. เรียกใช้ create_population_columns() เพื่อสร้างชื่อ column
3. ใช้ pd.read_csv() อ่านข้อมูลจาก URL โดยกำหนด

```python
sep="|" 

header=None 

names=population_columns 

dtype={"desc_ccaatt": "string"}
```

หลังจากสร้าง function แล้ว ให้ทดลองเรียกใช้กับข้อมูลกรุงเทพมหานคร ปี พ.ศ. 2568

`population_bkk_2568_df = read_population_by_age(2568, "10")`

จากนั้นทดลองเปลี่ยนเป็นจังหวัดอื่นหรือปีอื่น โดยเปลี่ยนแค่ argument ของ function เช่น

`population_other_df = read_population_by_age(2568, "50")`

In [27]:
# เขียนคำตอบของคุณใน Cell นี้

> ### เฉลยแบบฝึกหัดที่ 4
>
> การใช้ `str(...)[-2:]` ช่วยดึงเลขท้ายสองหลักของปี
>
> การใช้ `.zfill(2)` ช่วยให้รหัสจังหวัดมีสองหลัก เช่น `"1"` กลายเป็น `"01"`
>
> ไฟล์มีตัวคั่นท้ายบรรทัด จึงใช้ `usecols=range(len(population_columns))` เพื่อไม่อ่านคอลัมน์ว่างส่วนเกิน

In [58]:
def create_population_url(
    buddhist_year,
    province_code,
):
    year_suffix = str(
        buddhist_year
    )[-2:]

    province_code = str(
        province_code
    ).zfill(2)

    url = (
        "https://stat.bora.dopa.go.th/"
        f"new_stat/file/{year_suffix}12/"
        f"{year_suffix}12cc"
        f"{province_code}.txt"
    )

    return url

In [59]:
def create_population_columns():
    age_columns = []

    for age in range(102):
        age_columns.append(
            f"male_age_{age}"
        )

        age_columns.append(
            f"female_age_{age}"
        )

    summary_columns = [
        "male_tdob",
        "female_tdob",
        "total_tdob",
        "male_xhouse",
        "female_xhouse",
        "total_xhouse",
        "male_other_nat",
        "female_other_nat",
        "total_other_nat",
        "male_move",
        "female_move",
        "total_move",
        "male_tot",
        "female_tot",
        "total_tot",
    ]

    population_columns = (
        ["desc_ccaatt"]
        + age_columns
        + summary_columns
    )

    return population_columns

In [60]:
def read_population_by_age(
    buddhist_year,
    province_code,
):
    url = create_population_url(
        buddhist_year=buddhist_year,
        province_code=province_code,
    )

    population_columns = (
        create_population_columns()
    )

    dataframe = pd.read_csv(
        url,
        sep="|",
        header=None,
        names=population_columns,
        usecols=range(
            len(population_columns)
        ),
        dtype={
            "desc_ccaatt": "string",
        },
    )

    return dataframe

In [61]:
population_bkk_2568_df = (
    read_population_by_age(
        buddhist_year=2568,
        province_code="10",
    )
)

population_bkk_2568_df.head()

,desc_ccaatt,male_age_0,female_age_0,male_age_1,female_age_1,male_age_2,female_age_2,male_age_3,female_age_3,male_age_4,...,total_xhouse,male_other_nat,female_other_nat,total_other_nat,male_move,female_move,total_move,male_tot,female_tot,total_tot
0,กรุงเทพมหานคร,11640,10838,14323,13274,15588,14635,15814,14567,17733,...,73564,37804,28545,66349,1333,1150,2483,2526272,2896296,5422568
1,เขตพระนคร,52,40,55,47,73,48,70,64,78,...,436,388,256,644,1,3,4,18723,19685,38408
2,แขวงพระบรมมหาราชวัง,4,2,7,2,3,4,2,5,4,...,75,17,16,33,1,0,1,1684,1104,2788
3,แขวงวังบูรพาภิรมย์,10,9,12,10,20,8,14,15,26,...,62,110,90,200,0,0,0,4172,4066,8238
4,แขวงวัดราชบพิธ,5,6,4,2,5,3,1,3,1,...,6,24,23,47,0,0,0,1218,1509,2727


In [62]:
print(
    population_bkk_2568_df.shape
)

print(
    len(
        population_bkk_2568_df
        .columns
    )
)

(231, 220)
220


## แบบฝึกหัดที่ 5: สร้าง function กลางสำหรับนำเข้าข้อมูลตามประเภทไฟล์
หลังจากสร้าง function สำหรับอ่าน CSV และ Excel แล้ว ทีมต้องการ function 

กลางที่ช่วยเลือกวิธีนำเข้าไฟล์ให้เหมาะสมตามประเภทไฟล์

แนวคิดคือ
- ถ้าเป็นไฟล์ .csv ให้ใช้ function จากข้อ 2 คือ read_thai_csv()
- ถ้าเป็นไฟล์ .xlsx หรือ .xls ให้ใช้ function จากข้อ 3 คือ read_and_combine_excel_sheets()
- ถ้าเป็นประเภทไฟล์อื่น ให้แจ้งว่าไม่รองรับ

function ลักษณะนี้ช่วยให้ workflow การนำเข้าข้อมูลเป็นระบบมากขึ้น 

เพราะผู้ใช้ไม่ต้องจำว่าควรเรียก function ใดในแต่ละกรณี

ข้อกำหนด: 

ให้สร้าง function ชื่อ `import_data_by_file_type()` โดยมี parameter ดังนี้

```python
def import_data_by_file_type(
    file_path,
    file_type,
    selected_columns=None,
    code_columns=None,
    date_columns=None,
    sheet_list=None,
    dtype_dict=None
):
    ...
```

function นี้ต้องทำงานดังนี้
1. ถ้า `file_type == "csv"`

ให้เรียกใช้ `function read_thai_csv()` จากแบบฝึกหัดที่ 2 โดยส่ง `argument` ดังนี้

```python
read_thai_csv(
    file_path=file_path,
    selected_columns=selected_columns,
    code_columns=code_columns,
    date_columns=date_columns
)
```

แล้ว return DataFrame ที่อ่านได้

2. ถ้า `file_type == "excel"`

ให้เรียกใช้ function `read_and_combine_excel_sheets()` จากแบบฝึกหัดที่ 3 โดยส่ง `argument` ดังนี้

```python
read_and_combine_excel_sheets(
    file_path=file_path,
    sheet_list=sheet_list,
    selected_columns=selected_columns,
    dtype_dict=dtype_dict
)
```

แล้ว return DataFrame ที่อ่านได้

3. ถ้าเป็นประเภทไฟล์อื่น

ให้พิมพ์ข้อความ

```text
"Unsupported file type."
```

แล้ว return DataFrame ว่าง

`pd.DataFrame()`

In [28]:
# เขียนคำตอบของคุณใน Cell นี้

> ### เฉลยแบบฝึกหัดที่ 5
>
> ใช้ `.strip().lower()` ทำให้ค่าประเภทไฟล์อยู่ในรูปแบบมาตรฐานก่อนตรวจสอบ
>
> Function กลางทำหน้าที่เลือก Workflow ส่วนรายละเอียดการอ่านแต่ละประเภทอยู่ใน Function เฉพาะ

In [63]:
def import_data_by_file_type(
    file_path,
    file_type,
    selected_columns=None,
    code_columns=None,
    date_columns=None,
    sheet_list=None,
    dtype_dict=None,
):
    normalized_file_type = (
        file_type
        .strip()
        .lower()
    )

    if normalized_file_type == "csv":
        return read_mso_csv(
            file_path=file_path,
            selected_columns=(
                selected_columns
            ),
            code_columns=(
                code_columns or []
            ),
            date_columns=(
                date_columns or []
            ),
        )

    if normalized_file_type == "excel":
        return (
            read_and_combine_excel_sheets(
                file_path=file_path,
                sheet_list=(
                    sheet_list or []
                ),
                selected_columns=(
                    selected_columns
                ),
                dtype_dict=(
                    dtype_dict or {}
                ),
            )
        )

    print("Unsupported file type.")

    return pd.DataFrame()

In [64]:
csv_result_df = (
    import_data_by_file_type(
        file_path=mso_file,
        file_type="CSV",
        selected_columns=mso_usecols,
        code_columns=[
            "รหัสครัวเรือน",
            "รหัสประจำบ้าน",
            "รหัส cm",
        ],
        date_columns=[
            "วันที่สร้างครัวเรือน",
        ],
    )
)

csv_result_df.head()

,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,อายุ,เพศ,จังหวัด,ระดับการศึกษา,อาชีพหลัก,ประเภทกลุ่มเป้าหมาย,รหัส cm,หน่วยงานของ cm
0,648ff7d6b64d06b6b0dc7f02,<NA>,2023-06-19 13:38:14.881,3,หญิง,อุดรธานี,ไม่ได้เรียนหนังสือ,เกษตรกรรม (พืช ปศุสัตว์ ประมง),เด็กเล็ก,cm410011,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
1,62d3f7a6d6f101550054198b,70040223124,2022-07-17 18:51:02.889,55,หญิง,ราชบุรี,ไม่ได้เรียนหนังสือ,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm700018,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี
2,62a05c9df5674ecdd3805787,<NA>,2022-08-06 15:23:57.850,33,หญิง,หนองคาย,มัธยมศึกษาตอนต้น,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm430010,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
3,62908e378fa67bf5ad7d5555,<NA>,2022-05-27 15:39:19.197,5,หญิง,อุบลราชธานี,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm340009,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
4,6319b26ca3a37508384e36fc,<NA>,2022-08-09 16:14:20.652,3,ชาย,นครราชสีมา,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm300020,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา


In [65]:
excel_result_df = (
    import_data_by_file_type(
        file_path=excel_file,
        file_type="Excel",
        selected_columns=(
            farmer_usecols
        ),
        sheet_list=target_sheets,
        dtype_dict=farmer_dtype,
    )
)

excel_result_df.head()

,department_code,department,pid,province_code,province,amphur,tambon,is_farmer,farmer_type,main_occupation,income_in,income_out,debts_in,debts_out,updated_at,source_sheet
0,cpd,กรมส่งเสริมสหกรณ์,60e3f4907b85b690287f68a641b01b3d8392fbc28398f5...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
1,cpd,กรมส่งเสริมสหกรณ์,1ac1b0b4ad588a2c883621160bc0bbb5ca122dd42743b7...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
2,cpd,กรมส่งเสริมสหกรณ์,55e7b240314a3b5087b8ee4c376b52f229edda0d64761d...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
3,cpd,กรมส่งเสริมสหกรณ์,62a25e21b1ee1604c8cd4b812d81cf66854dd49cd25c6c...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile
4,cpd,กรมส่งเสริมสหกรณ์,4eb76a041e7389e4ae9d22ecefd36a5ac999f4c5ea5ebe...,<NA>,NaN,NaN,NaN,1,เกษตรกรผู้เป็นสมาชิกสหกรณ์และกลุุมเกษตรกร,NaN,NaN,NaN,NaN,NaN,NaN,v_cpd_fragile


In [66]:
unsupported_result_df = (
    import_data_by_file_type(
        file_path="data.json",
        file_type="json",
    )
)

unsupported_result_df

Unsupported file type.


""
